# Magnetism and Angular Momentum

## Purpose
In this lab you will:
- **Chemistry**: Understand how electron angular momentum creates magnetic properties, explore Zeeman splitting, and predict whether molecules are paramagnetic or diamagnetic
- **Coding**: Build interactive visualizations using `ipywidgets`, work with 3D plotting, and perform computational chemistry calculations with Psi4

**Real-world connection**: Magnetic properties of molecules are central to MRI imaging, magnetic materials design, and understanding electronic structure. The singlet-triplet gap we'll compute determines whether organic electronics materials can be used in displays and solar cells.

## Estimated Time: 75-90 minutes

## Success Criteria
- [ ] Created interactive visualization for Zeeman splitting
- [ ] Explored angular momentum coupling between L and S
- [ ] Calculated singlet-triplet gaps for three quinodimethane isomers
- [ ] Classified molecules as paramagnetic or diamagnetic
- [ ] Answered all reflection questions

---
# Libraries
Run this cell before proceeding. All imports for the entire notebook are consolidated here.

In [ ]:
# ============================================
# Standard scientific computing
# ============================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp

# ============================================
# Interactive widgets
# ============================================
import ipywidgets as widgets

# ============================================
# Specialized plotting
# ============================================
from matplotlib import cm
from scipy.special import sph_harm
from mpl_toolkits.mplot3d import Axes3D

# ============================================
# Computational chemistry
# ============================================
import psi4
import py3Dmol

# ============================================
# Helper functions (visualization code)
# ============================================
from magnetism_helper import (
    zeeman_plot,
    angular_momentum_coupling,
    interactive_coupling_plot,
    xyz_from_smiles,
    show_molecule,
    create_psi4_molecule
)

---
# Warmup: 3D Surface Plotting

The ability to visualize functions in three dimensions is a powerful tool for understanding physical chemistry. In this warmup, you'll create a 3D surface plot.

**Pattern**: Study the example, then create your own.

### Example: Plotting sin(x)cos(y)
Run these three cells to see how 3D plotting works.

In [ ]:
# ============================================
# SUBGOAL: Define the function symbolically
# ============================================
x, y = sp.symbols('x y')
f = sp.sin(x) * sp.cos(y)

# Convert to a numerical function
f_function = sp.lambdify((x, y), f, 'numpy')

In [ ]:
# ============================================
# SUBGOAL: Create a grid and evaluate
# ============================================
x_vals = np.linspace(-np.pi, np.pi, 100)
y_vals = np.linspace(-np.pi, np.pi, 100)
X, Y = np.meshgrid(x_vals, y_vals)
Z = f_function(X, Y)

In [ ]:
# ============================================
# SUBGOAL: Create the 3D visualization
# ============================================
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, cmap='viridis')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('f(x, y)')
ax.set_title('sin(x)cos(y)')
plt.tight_layout()
plt.show()

### CODE TASK (5 pts)
Create your own 3D plot using a different function. Try something like:
- `exp(-(x**2 + y**2))` (Gaussian)
- `sin(x**2 + y**2)` (radial waves)
- `x**2 - y**2` (saddle surface)

In [ ]:
# YOUR CODE HERE: Define your own function and create a 3D plot
# Follow the same three subgoals as the example above


---
# Part 1: Magnetism in the Bohr Model

The Bohr model treats electrons as orbiting the nucleus in circular paths with discrete angular momenta, $L = n\hbar$. We can derive the orbital magnetic dipole moment using classical electromagnetism.

A current loop produces a magnetic dipole moment $\mu = IA$. For an electron in a circular orbit:

$$\mu = -\frac{e}{2m_e}L$$

The potential energy of this magnetic dipole in a magnetic field $\vec{B}$ is:

$$U = \frac{e}{2m_e}\vec{L} \cdot \vec{B}$$

In **atomic units** ($e = m_e = 1$), with $L$ and $B$ aligned along the z-axis, this simplifies to:

$$U = \frac{1}{2}LB$$

### CODE TASK (10 pts)
1. Create a sympy function for potential energy $U(L, B)$ in atomic units
2. Make a 3D surface plot showing $U$ as a function of $L$ and $B$

**Hint**: Follow the warmup pattern - define symbols, create function, make meshgrid, plot.

In [ ]:
# ============================================
# SUBGOAL: Define the potential energy function
# ============================================
L, B = sp.symbols('L B')

# YOUR CODE HERE: Define U = (1/2) * L * B
U = # YOUR CODE HERE

# Convert to numerical function
U_function = sp.lambdify((L, B), U, 'numpy')

In [ ]:
# ============================================
# SUBGOAL: Create grid and visualize
# ============================================

# YOUR CODE HERE: Create meshgrid for L and B, evaluate U, create 3D plot
# Use L from -3 to 3 and B from 0 to 5 (Tesla)


### SHORT RESPONSE QUESTIONS

**Q1.1 (5 pts)**: From the graph you made, describe the relationship between $U$, $L$, and $B$. Is the relationship linear? What happens when $L$ or $B$ is zero?

**Q1.2 (5 pts)**: Why can we generalize this Bohr model equation to quantum mechanical orbitals? What property is conserved that makes this valid?

### ANSWERS

*Q1.1*: 

*Q1.2*: 

---
# Part 2: Zeeman Splitting

Atomic orbitals are described by spherical harmonics $Y_l^{m_l}(\theta, \phi)$, characterized by:
- **$l$**: angular momentum quantum number
- **$m_l$**: magnetic quantum number ($-l \leq m_l \leq l$)

The angular momentum magnitudes are:
$$L_{tot} = \sqrt{l(l+1)}\hbar, \quad L_z = m_l\hbar$$

Orbitals with the same $l$ are normally degenerate (same energy). However, an external magnetic field interacts with the magnetic moment, splitting these energy levels - this is the **Zeeman effect**.

You will now build an interactive visualization to explore this phenomenon.

### CODE TASK (15 pts)
Create an `interactive_zeeman_plot()` function that:
1. Uses `widgets.interact()` to make `zeeman_plot(l, ml, B)` interactive
2. Uses `widgets.IntSlider()` for $l$ (range 0-3) and $m_l$ (range -3 to 3)
3. Uses `widgets.FloatSlider()` for $B$ (range 0-6 Tesla)

**Target**: When complete, you should be able to adjust sliders and see the orbital shape and vector model update.

**Hint**: Look at `interactive_coupling_plot()` in the helper file for the pattern.

In [ ]:
def interactive_zeeman_plot():
    """
    Create an interactive widget to explore Zeeman splitting.
    
    Uses zeeman_plot(l, ml, B) with sliders for each parameter.
    """
    # ============================================
    # SUBGOAL: Define the slider widgets
    # ============================================
    l_widget = widgets.IntSlider(
        min=0, max=3, step=1, value=1, 
        description='l:'
    )
    
    # YOUR CODE HERE: Create ml_widget (IntSlider, -3 to 3)
    ml_widget = # YOUR CODE HERE
    
    # YOUR CODE HERE: Create B_widget (FloatSlider, 0 to 6)
    B_widget = # YOUR CODE HERE
    
    # ============================================
    # SUBGOAL: Connect widgets to plotting function
    # ============================================
    # YOUR CODE HERE: Use widgets.interact() with zeeman_plot


In [ ]:
# Run your interactive plot
interactive_zeeman_plot()

### EXPLORE
Use your interactive plot to investigate:
1. How the orbital shape changes with $l$ and $m_l$
2. How the vector model relates to the orbital shape
3. How the energy changes with $B$ and $m_l$

### SHORT RESPONSE QUESTIONS

**Q2.1 (5 pts)**: What is the relationship between the z-component of angular momentum ($L_z$) and the shape of the orbital? Consider the orientation relative to the z-axis.

**Q2.2 (5 pts)**: The x and y components of angular momentum are uncertain - we visualize this as precession around a cone. How does this relate to the toroidal (donut) shape of orbitals with $|m_l| > 0$?

**Q2.3 (5 pts)**: For wavefunctions with $m_l = 0$ and $l \geq 1$, the magnetic field causes no energy change. Explain why, despite the orbital having angular momentum ($L_{tot} \neq 0$).

### ANSWERS

*Q2.1*: 

*Q2.2*: 

*Q2.3*: 

---
# Part 3: Electron Spin and Total Angular Momentum

Electrons possess an intrinsic angular momentum called **spin**, with:
$$S = \sqrt{s(s+1)}\hbar, \quad S_z = m_s\hbar$$

where $s = 1/2$ for electrons, so $m_s = \pm 1/2$.

The total angular momentum $\vec{J}$ combines orbital and spin:
$$\vec{J} = \vec{L} + \vec{S}$$

This coupling leads to additional energy splitting in magnetic fields (spin-orbit coupling).

### EXPLORE TASK (5 pts)
Run `interactive_coupling_plot()` and explore:
1. How $\vec{J}$ relates to $\vec{L}$ and $\vec{S}$
2. What happens when $m_s$ changes sign (spin up vs spin down)
3. How the allowed values of $J_z$ depend on $m_l$ and $m_s$

In [ ]:
# Run the provided interactive coupling visualization
interactive_coupling_plot()

### SHORT RESPONSE QUESTIONS

**Q3.1 (5 pts)**: Based on your exploration, how is the quantum number $j$ related to $l$ and $s$? What are the possible values of $j$ for a given $l$?

**Q3.2 (5 pts)**: How does $J_z$ (the z-projection of total angular momentum) depend on $S_z$ and $L_z$? Is this additive, multiplicative, or something else?

**Q3.3 (5 pts)**: For an electron in one of the three $p$ orbitals ($l=1$, so $m_l = -1, 0, +1$) with either $m_s = +1/2$ or $m_s = -1/2$, list all possible values of $J_z$. How many unique values are there?

### ANSWERS

*Q3.1*: 

*Q3.2*: 

*Q3.3*: 

---
# Part 4: Probing Organic Diradicals with Psi4

Organic diradicals contain two unpaired electrons and exhibit magnetic properties determined by their **singlet-triplet energy gap** ($\Delta E_{ST}$):

- **Singlet state** ($S=0$): electrons paired, **diamagnetic** (repelled by magnetic field)
- **Triplet state** ($S=1$): electrons unpaired, **paramagnetic** (attracted to magnetic field)

We'll compute $\Delta E_{ST}$ for three quinodimethane isomers using Psi4:

| Molecule | Abbreviation | Position of CH₂ groups |
|----------|--------------|------------------------|
| ortho-quinodimethane | oqdm | 1,2 (adjacent) |
| meta-quinodimethane | mqdm | 1,3 (one apart) |
| para-quinodimethane | pqdm | 1,4 (opposite) |

### SMILES Notation

The SMILES strings for these diradicals use `[C]` to indicate a radical carbon:

```python
# ortho-quinodimethane (1,2-positions)
oqdm = "C1=CC=CC([C]([H])[H])=C1[C]([H])[H]"

# meta-quinodimethane (1,3-positions)  
mqdm = "C1=CC(=CC([C]([H])[H])=C1)[C]([H])[H]"

# para-quinodimethane (1,4-positions)
pqdm = "C1=CC(=CC=C1[C]([H])[H])[C]([H])[H]"
```

### CODE TASK (20 pts)
1. Define SMILES strings for all three molecules
2. Display each molecule using `show_molecule()`
3. Calculate singlet and triplet energies for each molecule
4. Compute $\Delta E_{ST}$ = $E_{singlet} - E_{triplet}$ in kJ/mol

**Key settings**:
- For singlet: use `brokensymmetry=True` and `spin_multiplicity=1`
- For triplet: use `spin_multiplicity=3` (no broken symmetry needed)
- Conversion: 1 Hartree = 2625 kJ/mol

In [ ]:
# ============================================
# SUBGOAL: Define the molecules
# ============================================

# ortho-quinodimethane
oqdm = "C1=CC=CC([C]([H])[H])=C1[C]([H])[H]"

# YOUR CODE HERE: Define mqdm (meta-quinodimethane)
mqdm = # YOUR CODE HERE

# YOUR CODE HERE: Define pqdm (para-quinodimethane)
pqdm = # YOUR CODE HERE

In [ ]:
# ============================================
# SUBGOAL: Visualize the molecules
# ============================================
print("ortho-quinodimethane:")
show_molecule(oqdm)

In [ ]:
# YOUR CODE HERE: Display mqdm
print("meta-quinodimethane:")


In [ ]:
# YOUR CODE HERE: Display pqdm
print("para-quinodimethane:")


In [ ]:
# ============================================
# SUBGOAL: Configure Psi4 calculation
# ============================================
# These settings should not be changed
psi4.set_memory('4 GB')
my_theory = 'B3LYP/6-31G(d)'
psi4.set_options({
    "scf__reference": "uhf",
})

### Example: Calculating $\Delta E_{ST}$ for oqdm

In [ ]:
# ============================================
# SUBGOAL: Calculate singlet energy (broken symmetry)
# ============================================
psi4.set_output_file('oqdm.dat', False)

p4oqdm_s = create_psi4_molecule(oqdm, spin_multiplicity=1)
oqdm_s_energy, oqdm_s_wfn = psi4.energy(
    my_theory, 
    return_wfn=True,
    molecule=p4oqdm_s,
    brokensymmetry=True
)
print(f"oqdm singlet energy: {oqdm_s_energy:.6f} Hartree")

In [ ]:
# ============================================
# SUBGOAL: Calculate triplet energy
# ============================================
p4oqdm_t = create_psi4_molecule(oqdm, spin_multiplicity=3)
oqdm_t_energy, oqdm_t_wfn = psi4.energy(
    my_theory, 
    return_wfn=True,
    molecule=p4oqdm_t
)
print(f"oqdm triplet energy: {oqdm_t_energy:.6f} Hartree")

In [ ]:
# ============================================
# SUBGOAL: Calculate singlet-triplet gap
# ============================================
oqdm_delta_e_st = (oqdm_s_energy - oqdm_t_energy) * 2625  # Convert to kJ/mol
print(f"oqdm ΔE_ST = {oqdm_delta_e_st:.2f} kJ/mol")

if oqdm_delta_e_st > 0:
    print("Ground state: TRIPLET (paramagnetic)")
else:
    print("Ground state: SINGLET (diamagnetic)")

### Your Turn: Calculate $\Delta E_{ST}$ for mqdm and pqdm

In [ ]:
# YOUR CODE HERE: Calculate singlet and triplet energies for mqdm
# Follow the same pattern as oqdm above

psi4.set_output_file('mqdm.dat', False)

# Singlet calculation

# Triplet calculation

# Calculate and print delta E_ST


In [ ]:
# YOUR CODE HERE: Calculate singlet and triplet energies for pqdm
# Follow the same pattern as oqdm above

psi4.set_output_file('pqdm.dat', False)

# Singlet calculation

# Triplet calculation

# Calculate and print delta E_ST


### Results Summary
Fill in your calculated values:

In [ ]:
# Create a summary table of your results
results = pd.DataFrame({
    'Molecule': ['oqdm', 'mqdm', 'pqdm'],
    'ΔE_ST (kJ/mol)': [oqdm_delta_e_st, 0.0, 0.0],  # Replace 0.0 with your values
    'Ground State': ['', '', ''],  # Fill in: 'singlet' or 'triplet'
    'Magnetic Property': ['', '', '']  # Fill in: 'diamagnetic' or 'paramagnetic'
})
results

### SHORT RESPONSE QUESTIONS

**Q4.1 (5 pts)**: Rank the three molecules in order of increasing $|\Delta E_{ST}|$. Which molecule(s), if any, have a triplet ground state?

**Q4.2 (5 pts)**: Classify each molecule as paramagnetic or diamagnetic based on its ground state.

### ANSWERS

*Q4.1*: 

*Q4.2*: 

---
# Reflection

These questions ask you to synthesize what you learned across the entire lab.

### SHORT RESPONSE QUESTIONS

**Q5.1 (10 pts)**: How does the quantization of angular momentum in atomic orbitals influence the interaction of electrons with magnetic fields? Specifically, explain:
- Why only certain energy shifts are observed (discrete splitting)
- How this is observed experimentally in an atomic spectrum (Zeeman effect)

**Q5.2 (10 pts)**: Describe how we computationally tested whether the quinodimethane molecules are paramagnetic or diamagnetic. Why is this distinction important for applications like:
- MRI contrast agents
- Organic electronics
- Magnetic materials

### ANSWERS

*Q5.1*: 

*Q5.2*: 

---
# References

1. [Physics Libretexts - Orbital Magnetic Dipole Moment of the Electron](https://phys.libretexts.org/Bookshelves/University_Physics/University_Physics_(OpenStax)/University_Physics_III_-_Optics_and_Modern_Physics_(OpenStax)/08%3A_Atomic_Structure/8.03%3A_Orbital_Magnetic_Dipole_Moment_of_the_Electron)

2. [Physics Libretexts - Electron Spin](https://phys.libretexts.org/Courses/Georgia_State_University/GSU-TM-Physics_II_(2212)/13%3A_Atomic_Structure/13.03%3A_Electron_Spin)

3. [Chem Libretexts - Total Angular Momentum](https://chem.libretexts.org/Courses/Pacific_Union_College/Quantum_Chemistry/08%3A_Multielectron_Atoms/8.09%3A_The_Allowed_Values_of_J_-_the_Total_Angular_Momentum_Quantum_Number)

4. [Psi4 Manual - SCF Methods](https://psicode.org/psi4manual/4.0b5/scf)